# Build doublet datasets at DSB cutoffs 4-9

Replays the production doublet-calling chain from `processing/scanpy_processing_SIG13_clean.ipynb`
with the DSB UMI threshold as a parameter, and writes one dataset per cutoff.

In [ ]:
# Repo-relative paths: inputs are read from imports_stable/, outputs are written to analysis_outs/
import os
from pathlib import Path
_here = Path.cwd().resolve()
REPO_DIR = str(next(p for p in [_here, *_here.parents] if (p / "imports_stable").is_dir()))
IMPORTS_DIR = f"{REPO_DIR}/imports_stable"
OUTS_DIR = f"{REPO_DIR}/analysis_outs/02_combinatorial_screen_signalseq_SIG13"
for _d in ["qc_barcode_cutoff", "scanpy_outs/cutoff_sweep"]:
    os.makedirs(f"{OUTS_DIR}/{_d}", exist_ok=True)

In [1]:
import h5py
import numpy as np
import pandas as pd
import anndata as ad
from anndata.io import read_elem

H5MU = IMPORTS_DIR + "/SIG13/scanpy_outs/SIG13_full_bc_processed.h5mu"
PROD_H5AD = IMPORTS_DIR + "/SIG13/scanpy_outs/SIG13_doublets_DSB7.h5ad"
SPCA_LOADINGS = IMPORTS_DIR + "/SIG13/analysis_outs/spca/zscore_degs_allLigands_0.1_alpha1.0_sPCA_loadings.csv"

H5AD_DIR = OUTS_DIR + "/scanpy_outs/cutoff_sweep"
OUT_DIR = OUTS_DIR + "/qc_barcode_cutoff"

CUTOFFS = list(range(1, 11))
REF_CUTOFF = 7
# DSB7 reuses the production GLM output, so no counts file is written for it
GLM_CUTOFFS = [k for k in CUTOFFS if k != REF_CUTOFF]

import os
os.makedirs(H5AD_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

## Calling helpers

Copied unchanged from the production processing notebook.

In [2]:
def generate_feature_calls(adata, umi_threshold=2):
    """Per-cell barcode calls at a DSB threshold: which features, their scores, how many."""
    X = adata.X
    if hasattr(X, "tocoo"):
        rows, cols = (X >= umi_threshold).nonzero()
        umi_counts = X[rows, cols].A1
    else:
        rows, cols = (X >= umi_threshold).nonzero()
        umi_counts = X[rows, cols]

    data = pd.DataFrame({
        "cell_barcode": adata.obs.index[rows],
        "feature_name": adata.var.index[cols],
        "umi_count": umi_counts,
    })
    grouped = data.groupby("cell_barcode").agg({
        "feature_name": lambda x: "|".join(x),
        "umi_count": lambda x: "|".join(map(str, x)),
    }).reset_index()
    grouped["num_features"] = grouped["feature_name"].str.count(r"\|") + 1
    return grouped.rename(columns={"feature_name": "feature_call", "umi_count": "num_umis"})


def process_feature_call(df, feature_call_col="feature_call"):
    """Split each feature call into its ligand identities and its round1/round2 groups."""
    split_calls = df[feature_call_col].fillna("").str.split("|")

    ligand_calls, group_calls = [], []
    for calls in split_calls:
        if calls == [""]:
            ligand_calls.append("")
            group_calls.append("")
        else:
            ligand_calls.append("_".join([c.rsplit("_", 1)[0] for c in calls]))
            group_calls.append("_".join(sorted([c.rsplit("_", 1)[-1] for c in calls])))

    df["ligand_call"] = ligand_calls
    df["group_call"] = group_calls
    return df

## Load barcodes and metadata

In [3]:
f = h5py.File(H5MU, "r")
bc = read_elem(f["mod"]["bc"])
obs_all = read_elem(f["mod"]["rna"]["obs"])
var = read_elem(f["mod"]["rna"]["var"])
print(bc.shape, obs_all.shape)

(628244, 74) (628244, 42)


## Doublet population at one cutoff

Five steps, in production order: call barcodes, keep correct doublets (one round1 + one round2),
drop protein ligands paired with `linker6-9_round2`, drop replicate x feature-call combinations
with fewer than 10 cells, simplify linker names.

In [4]:
def build_cutoff_obs(cutoff):
    calls = process_feature_call(generate_feature_calls(bc, umi_threshold=cutoff))
    calls = calls.set_index("cell_barcode").reindex(obs_all.index)

    obs = obs_all.copy()
    for col in ["num_features", "feature_call", "num_umis", "ligand_call", "group_call"]:
        obs[col] = calls[col]
    obs["num_features"] = obs["num_features"].fillna(0)

    # correct doublets only
    obs = obs[(obs["num_features"] == 2) & (obs["group_call"] == "round1_round2")].copy()

    feature_call = obs["feature_call"].astype(str)
    possible = ~feature_call.str.fullmatch(
        r"^(?!linker[23]_round1\|)[^|]+_round1\|linker[6789]_round2$"
    )
    obs["replicate_feature_call"] = obs["replicate"].astype(str) + "_" + feature_call

    control_num = (obs[possible]
                   .groupby(["replicate", "feature_call", "replicate_feature_call"], observed=True)
                   .size().reset_index(name="n_cells"))
    inappropriate = control_num.query("n_cells < 10")["replicate_feature_call"].unique().astype(str)

    obs = obs[possible & ~obs["replicate_feature_call"].isin(inappropriate)].copy()
    obs["ligand_call"] = obs["ligand_call"].str.replace(r"linker[1-9]", "linker", regex=True)
    obs[["ligand_call_round1", "ligand_call_round2"]] = obs["ligand_call"].str.split("_", expand=True)
    obs["dsb_cutoff"] = cutoff
    return obs

In [5]:
cutoff_obs = {k: build_cutoff_obs(k) for k in CUTOFFS}
pd.Series({k: len(v) for k, v in cutoff_obs.items()}, name="n_cells").to_frame()

,n_cells
1,81057
2,160182
3,251859
4,327091
5,369315
6,374989
7,349678
8,300300
9,236273
10,169075


## Validation

The regenerated DSB7 population must be exactly the production dataset, and the condition set
must not move across cutoffs - both are assumptions the downstream comparisons rest on.

In [6]:
with h5py.File(PROD_H5AD, "r") as h:
    prod_cells = pd.Index(read_elem(h["obs"]["_index"]))

assert cutoff_obs[REF_CUTOFF].index.equals(prod_cells), "regenerated DSB7 != production DSB7"
print(f"DSB7 matches production dataset exactly ({len(prod_cells)} cells)")

# Conditions are identical across DSB1-9; DSB10 drops a couple below the 10-cell filter, so the
# common sets are computed and written out rather than asserted, and everything downstream
# restricts to them.
cond_sets = {k: set(v["ligand_call"].unique()) for k, v in cutoff_obs.items()}
unit_sets = {k: set(v["ligand_call"] + "__" + v["replicate"].astype(str)) for k, v in cutoff_obs.items()}

common_conditions = set.intersection(*cond_sets.values())
common_units = set.intersection(*unit_sets.values())

for k in CUTOFFS:
    missing = sorted(cond_sets[REF_CUTOFF] - cond_sets[k])
    print(f"DSB{k}: {len(cond_sets[k])} conditions, {len(unit_sets[k])} units"
          + (f"  missing vs DSB7: {missing}" if missing else ""))

print(f"\ncommon across all cutoffs: {len(common_conditions)} conditions, {len(common_units)} units")

pd.Series(sorted(common_conditions), name="ligand_call").to_csv(
    f"{OUT_DIR}/cutoff_common_conditions.csv", index=False)
pd.Series(sorted(common_units), name="unit").to_csv(
    f"{OUT_DIR}/cutoff_common_units.csv", index=False)

DSB7 matches production dataset exactly (349678 cells)


DSB1: 660 conditions, 1314 units
DSB2: 660 conditions, 1320 units
DSB3: 660 conditions, 1320 units
DSB4: 660 conditions, 1320 units
DSB5: 660 conditions, 1320 units
DSB6: 660 conditions, 1320 units
DSB7: 660 conditions, 1320 units
DSB8: 660 conditions, 1320 units
DSB9: 660 conditions, 1318 units
DSB10: 658 conditions, 1309 units  missing vs DSB7: ['IL23_IL1A', 'IL23_IL9']

common across all cutoffs: 658 conditions, 1303 units


## Summary table

In [7]:
ref_cells = set(cutoff_obs[REF_CUTOFF].index)

rows = []
for k, obs in cutoff_obs.items():
    n_per_cond = obs.groupby("ligand_call", observed=True).size()
    shared = len(ref_cells & set(obs.index))
    union = len(ref_cells | set(obs.index))
    rows.append({
        "cutoff": k,
        "n_cells": len(obs),
        "n_conditions": obs["ligand_call"].nunique(),
        "n_units": len(unit_sets[k]),
        "median_cells_per_condition": n_per_cond.median(),
        "min_cells_per_condition": n_per_cond.min(),
        "mean_total_counts": obs["total_counts"].mean(),
        "mean_n_genes": obs["n_genes_by_counts"].mean(),
        "n_shared_DSB7": shared,
        "jaccard_DSB7": shared / union,
    })

summary = pd.DataFrame(rows)
summary.to_csv(f"{OUT_DIR}/cutoff_dataset_summary.csv", index=False)
summary

,cutoff,n_cells,n_conditions,n_units,median_cells_per_condition,min_cells_per_condition,mean_total_counts,mean_n_genes,n_shared_DSB7,jaccard_DSB7
0,1,81057,660,1314,102.5,15,6403.433594,2949.574855,49251,0.129104
1,2,160182,660,1320,199.0,51,6742.634277,3052.075570,103654,0.255176
2,3,251859,660,1320,312.5,85,7069.520996,3144.618290,173749,0.406157
3,4,327091,660,1320,406.0,99,7359.060059,3223.530091,240710,0.552012
4,5,369315,660,1320,462.0,93,7623.784668,3294.622750,292452,0.685636
5,6,374989,660,1320,473.0,82,7905.088867,3368.434770,327119,0.822842
6,7,349678,660,1320,453.0,62,8231.306641,3452.515120,349678,1.000000
7,8,300300,660,1320,393.0,42,8633.194336,3552.288105,287400,0.792657
8,9,236273,660,1318,311.0,24,9105.505859,3663.306396,218519,0.594720
9,10,169075,658,1309,219.0,10,9588.547852,3769.730735,151722,0.413377


## Per-cutoff counts files for the GLM

`counts` layer only - the GLM needs nothing else, and dropping the other three layers and the
embeddings keeps each file around a quarter of the production 39 GB.

In [8]:
GLM_OBS_COLS = [
    "ligand_call", "ligand_call_round1", "ligand_call_round2", "feature_call", "num_umis",
    "lane", "replicate", "pct_counts_mt", "S_score", "G2M_score",
    "total_counts", "n_genes_by_counts", "log1p_total_counts", "log1p_total_counts_bc",
    "dsb_cutoff",
]

counts = read_elem(f["mod"]["rna"]["layers"]["counts"])
print(counts.shape, counts.dtype)

(628244, 13050) float32


In [9]:
row_of = pd.Series(np.arange(len(obs_all)), index=obs_all.index)

for k in GLM_CUTOFFS:
    obs = cutoff_obs[k][GLM_OBS_COLS].copy()
    adata = ad.AnnData(obs=obs, var=var.copy(),
                       layers={"counts": counts[row_of[obs.index].values]})
    path = f"{H5AD_DIR}/SIG13_doublets_DSB{k}.h5ad"
    adata.write_h5ad(path)
    print(f"wrote {path}  {adata.shape}")
    del adata

wrote /data1/rudenska/EYW/SIG13/scanpy_outs/cutoff_sweep/SIG13_doublets_DSB1.h5ad  (81057, 13050)


wrote /data1/rudenska/EYW/SIG13/scanpy_outs/cutoff_sweep/SIG13_doublets_DSB2.h5ad  (160182, 13050)


wrote /data1/rudenska/EYW/SIG13/scanpy_outs/cutoff_sweep/SIG13_doublets_DSB3.h5ad  (251859, 13050)


wrote /data1/rudenska/EYW/SIG13/scanpy_outs/cutoff_sweep/SIG13_doublets_DSB4.h5ad  (327091, 13050)


wrote /data1/rudenska/EYW/SIG13/scanpy_outs/cutoff_sweep/SIG13_doublets_DSB5.h5ad  (369315, 13050)


wrote /data1/rudenska/EYW/SIG13/scanpy_outs/cutoff_sweep/SIG13_doublets_DSB6.h5ad  (374989, 13050)


wrote /data1/rudenska/EYW/SIG13/scanpy_outs/cutoff_sweep/SIG13_doublets_DSB8.h5ad  (300300, 13050)


wrote /data1/rudenska/EYW/SIG13/scanpy_outs/cutoff_sweep/SIG13_doublets_DSB9.h5ad  (236273, 13050)


wrote /data1/rudenska/EYW/SIG13/scanpy_outs/cutoff_sweep/SIG13_doublets_DSB10.h5ad  (169075, 13050)


## Shared file for the sPCA scoring

All cells that are a called doublet at any cutoff, restricted to the sPCA genes, `log1p_norm`
in `X`, with each cutoff's condition call and membership flag in `.obs`.

In [10]:
loadings = pd.read_csv(SPCA_LOADINGS)
comp_genes = [g for g in loadings["gene"].unique() if g in var.index]
print(f"{len(comp_genes)} of {loadings['gene'].nunique()} component genes present")

spca_cells = sorted(set().union(*[set(v.index) for v in cutoff_obs.values()]), key=row_of.get)
print(f"{len(spca_cells)} cells called a doublet at >= 1 cutoff")

6610 of 6610 component genes present


503574 cells called a doublet at >= 1 cutoff


In [11]:
log1p = read_elem(f["mod"]["rna"]["layers"]["log1p_norm"])
gene_idx = pd.Series(np.arange(len(var)), index=var.index)[comp_genes].values
cell_idx = row_of[spca_cells].values

spca_obs = obs_all.loc[spca_cells, ["lane", "replicate", "total_counts",
                                    "n_genes_by_counts", "pct_counts_mt"]].copy()
for k in CUTOFFS:
    obs = cutoff_obs[k]
    spca_obs[f"keep_DSB{k}"] = spca_obs.index.isin(obs.index)
    spca_obs[f"ligand_call_DSB{k}"] = obs["ligand_call"].reindex(spca_obs.index)
    spca_obs[f"num_umis_DSB{k}"] = obs["num_umis"].reindex(spca_obs.index)

spca = ad.AnnData(X=log1p[cell_idx][:, gene_idx], obs=spca_obs, var=var.loc[comp_genes].copy())
path = f"{H5AD_DIR}/SIG13_cutoffSweep_spcaGenes_log1pnorm.h5ad"
spca.write_h5ad(path)
print(f"wrote {path}  {spca.shape}")

wrote /data1/rudenska/EYW/SIG13/scanpy_outs/cutoff_sweep/SIG13_cutoffSweep_spcaGenes_log1pnorm.h5ad  (503574, 6610)
